# Тема 4. Инструменты и агент на LangChain / LangGraph

**Модуль II · дополнительный пример**

В основных примерах модуля агент собран на «голом» OpenAI SDK — чтобы механика была видна пошагово. Здесь — **идиоматичный** способ на **LangChain** и **LangGraph**. Это тот же цикл «модель ⇄ инструменты», но на фреймворке, который берёт на себя рутину.

### Две библиотеки — общая картина (если вы видите их впервые)

- **LangChain** — набор *строительных блоков* для приложений с LLM. Нам нужны четыре:
  - **модель** — класс `ChatOpenAI` (единый интерфейс к любой OpenAI-совместимой модели);
  - **инструменты** — обычные Python-функции, обёрнутые декоратором `@tool`;
  - **сообщения** — объекты `SystemMessage` / `HumanMessage` / `AIMessage` / `ToolMessage` (роли в диалоге);
  - **готовый агент** — функция `create_agent` собирает агента «под ключ».
- **LangGraph** — *оркестратор*: он описывает работу агента как **граф** из трёх сущностей:
  - **вершины (nodes)** — шаги (например, «вызвать модель», «выполнить инструмент»);
  - **рёбра (edges)** — переходы между шагами (в т. ч. **условные** — ветвление);
  - **состояние (state)** — общие данные, которые вершины читают и дополняют.
  LangGraph гоняет по этому графу цикл «модель → инструменты → модель → …», пока агент не выдаст финальный ответ.

**Как читать ноутбук.** Каждый метод, декоратор и функция объясняются в markdown-ячейке *перед* кодом, где встречаются впервые. В конце — сводная таблица-справочник по всем использованным именам.

### Доступ к модели
Параметры — из переменных окружения: `LLM_API_KEY`, `LLM_BASE_URL`, `LLM_MODEL`.

## Модель: класс `ChatOpenAI` (LangChain)

`ChatOpenAI` из пакета `langchain_openai` — это обёртка над chat-эндпоинтом любой **OpenAI-совместимой** модели. Она даёт единый интерфейс: один и тот же объект `llm` умеет и просто отвечать текстом, и вызывать инструменты, и подключаться к графу.

**Аргументы конструктора, которые используем:**

| Аргумент | Что задаёт |
|----------|-----------|
| `model` | идентификатор модели (напр. `openai/gpt-4.1-mini`) |
| `base_url` | адрес OpenAI-совместимого эндпоинта (провайдер) |
| `api_key` | ключ доступа (берём из переменной окружения — в код не вставляем) |
| `temperature` | «разброс» ответов; `0` — максимально детерминированно (важно для воспроизводимости) |

**Методы объекта `llm`, которые понадобятся дальше:**

- `llm.invoke(messages)` — отправить список сообщений и получить один ответ (объект `AIMessage`).
- `llm.bind_tools(tools)` — вернуть *копию* модели, которая «знает» о наборе инструментов (передаёт их JSON-схемы модели, чтобы та могла их запрашивать).
- `llm.model_name` — атрибут с именем модели (для печати/проверки).

Ниже мы только *создаём* объект модели; вызывать его будем в следующих разделах.

In [1]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=os.environ.get("LLM_MODEL", "openai/gpt-4.1-mini"),
                 base_url=os.environ.get("LLM_BASE_URL"),
                 api_key=os.environ.get("LLM_API_KEY"), temperature=0)
print("Модель:", llm.model_name)

Модель: openai/gpt-4.1-mini


## Инструменты через декоратор `@tool`

`@tool` (из `langchain_core.tools`) — декоратор. Он **не меняет** тело вашей функции, а *оборачивает* её в объект-инструмент (`BaseTool`), который LangChain и агент понимают. Всё, что нужно модели для вызова, собирается **автоматически из самой функции**:

| Что нужно модели | Откуда берётся |
|------------------|----------------|
| **имя** инструмента | имя функции (`search_products`) |
| **описание** (по нему модель решает, *когда* звать инструмент) | докстринг `"""..."""` — поэтому его пишут содержательно |
| **схема аргументов** (какие поля и типы) | аннотации типов (`query: str`) → JSON Schema через Pydantic |

Писать JSON Schema руками не нужно — в этом весь смысл `@tool`.

**Важно понимать поток управления:** модель никогда сама не выполняет ваш Python. Она лишь *возвращает запрос* вида «вызови `search_products` с `query='USB-C hub'`». Собственно запуск функции делает агент (или узел `ToolNode` дальше), а результат возвращается модели как отдельное сообщение.

У полученного объекта есть атрибуты для интроспекции: `.name`, `.description`, `.args` (схема аргументов). Список таких объектов (`TOOLS`) мы затем передаём агенту или привязываем к модели.

In [2]:
from langchain_core.tools import tool

CATALOG = {"P-100": ("Wireless Headphones", 4990), "P-300": ("USB-C Hub", 2490)}
ORDERS = {"ORD-1002": {"sku": "P-300", "qty": 2, "status": "shipped"}}

@tool
def search_products(query: str) -> list[dict]:
    """Найти товары по ключевому слову в названии (каталог на английском)."""
    return [{"sku": s, "name": n, "price": p} for s, (n, p) in CATALOG.items()
            if query.lower() in n.lower()]

@tool
def get_order(order_id: str) -> dict:
    """Получить информацию о заказе по идентификатору."""
    return {"order_id": order_id, **ORDERS.get(order_id, {"error": "not_found"})}

TOOLS = [search_products, get_order]
print("Инструмент:", search_products.name, "| схема аргументов:", search_products.args)

Инструмент: search_products | схема аргументов: {'query': {'title': 'Query', 'type': 'string'}}


## Готовый агент: функция `create_agent`

`create_agent(model, tools, ...)` из `langchain.agents` собирает **ReAct-агента** «под ключ»: внутри это уже готовый граф LangGraph, реализующий цикл «модель решает → вызвать инструмент → отдать результат модели → повторить, пока не готов ответ». Вам не нужно строить этот цикл руками.

**Что принимает:**
- `model` — объект модели (`llm`);
- `tools` — список инструментов (наши `@tool`-функции);
- `system_prompt=` (необязательно) — системная инструкция агенту (используем в модуле IV).

**Что возвращает:** *скомпилированный граф* — объект с методами `.invoke(...)` и `.stream(...)` (те же, что у любого графа LangGraph).

**Формат ввода/вывода `invoke`:**
- на вход — словарь `{"messages": [...]}`; сообщение можно задать кортежем `("user", "текст")` (роль, текст) или объектом сообщения;
- на выходе — словарь, где `result["messages"]` это **вся трасса** диалога: исходный запрос, обращения модели к инструментам, ответы инструментов и финальный ответ. Поэтому в трассе из простого вопроса получается несколько сообщений, а `result["messages"][-1].content` — это финальный текст для пользователя.

In [3]:
from langchain.agents import create_agent

agent = create_agent(llm, TOOLS)
result = agent.invoke({"messages": [("user", "Есть ли USB-C хаб и какой статус заказа ORD-1002?")]})
print("Сообщений в трассе:", len(result["messages"]))
print("Ответ:", result["messages"][-1].content[:300])

Сообщений в трассе: 5
Ответ: Да, в наличии есть USB-C хаб по цене 2490 рублей. Ваш заказ ORD-1002 на 2 таких хаба уже отправлен. Если нужна дополнительная информация, дайте знать!


## Кастомный граф LangGraph (устройство «изнутри»)

`create_agent` скрыл цикл внутри. Теперь соберём тот же цикл **вручную**, чтобы увидеть все детали LangGraph. Разберём каждое имя из кода ниже.

**Состояние (state).**
- `MessagesState` — готовая схема состояния из `langgraph.graph`. Это, по сути, словарь с единственным полем `messages: list`. У этого поля встроенный **reducer**: когда вершина возвращает `{"messages": [новое]}`, LangGraph **добавляет** его к списку, а не перезаписывает. Благодаря этому история диалога накапливается сама.

**Вершины (nodes).** Вершина — это обычная функция `state -> частичный_словарь_состояния`. Она читает нужные поля из `state` и возвращает только то, что хочет изменить (LangGraph сам вольёт это в состояние через reducer).
- `agent_node(state)` — наша вершина: берёт накопленные `messages`, добавляет системное сообщение и зовёт `llm_with_tools.invoke(...)`. Возвращает ответ модели.
- `ToolNode(TOOLS)` — **готовая** вершина из `langgraph.prebuilt`. Она смотрит на последнее сообщение модели, находит в нём запросы на вызовы инструментов, **исполняет** соответствующие функции из `TOOLS` и возвращает их результаты как `ToolMessage`. Руками парсить вызовы не нужно.

**Модель с инструментами.**
- `llm.bind_tools(TOOLS)` — даёт модель, которая знает схемы инструментов и потому *может* их запрашивать. (У `create_agent` это делается внутри; здесь — явно.)

**Сборка графа.**
- `StateGraph(MessagesState)` — «строитель» графа поверх выбранной схемы состояния.
- `g.add_node("имя", функция)` — зарегистрировать вершину под именем.
- `START`, `END` — специальные метки-вершины: точка входа и точка выхода графа.
- `g.add_edge(A, B)` — **безусловное** ребро: после `A` всегда идём в `B`. Здесь: `START→agent` (вход) и `tools→agent` (после инструментов — снова к модели, замыкаем цикл).
- `g.add_conditional_edges("agent", tools_condition)` — **условное** ребро: функция-маршрутизатор `tools_condition` (готовая, из `langgraph.prebuilt`) смотрит на ответ модели: если та запросила инструмент → идём в вершину `"tools"`, иначе → `END` (ответ готов). Это и есть развилка ReAct-цикла.
- `g.compile()` — «заморозить» описание в исполняемый граф (объект с `.invoke` / `.stream`).

**Запуск.**
- `graph.stream(вход)` — выполнить граф, **отдавая обновления по мере прохождения вершин** (удобно, чтобы наблюдать цикл шаг за шагом). Каждый элемент — словарь `{имя_вершины: обновление_состояния}`.
- (`graph.invoke(вход)` — то же, но без промежуточных шагов: вернёт только финальное состояние.)

**Сообщения.**
- `SystemMessage("...")` — сообщение с ролью *system* (инструкция модели). Аналогично есть `HumanMessage` (пользователь) и `ToolMessage` (результат инструмента); у объекта сообщения есть `.type`, `.content`, а у ответа модели — `.tool_calls`.

In [4]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import SystemMessage

llm_with_tools = llm.bind_tools(TOOLS)
SYS = SystemMessage("Ты — ассистент интернет-магазина. Каталог на английском.")

def agent_node(state: MessagesState):
    return {"messages": [llm_with_tools.invoke([SYS] + state["messages"])]}

g = StateGraph(MessagesState)
g.add_node("agent", agent_node)
g.add_node("tools", ToolNode(TOOLS))
g.add_edge(START, "agent")
g.add_conditional_edges("agent", tools_condition)   # -> "tools" или END
g.add_edge("tools", "agent")
graph = g.compile()

# Стримим шаги, чтобы увидеть цикл «модель -> инструменты -> модель»
for step in graph.stream({"messages": [("user", "Найди USB-C хаб в каталоге")]}):
    for node, upd in step.items():
        last = upd["messages"][-1]
        tag = "tool" if last.type == "tool" else ("assistant->tool_call" if getattr(last, "tool_calls", None) else "assistant")
        print(f"[{node}] {tag}: {str(getattr(last,'content','') or last.tool_calls)[:90]}")

[agent] assistant->tool_call: [{'name': 'search_products', 'args': {'query': 'USB-C hub'}, 'id': 'call_1pU7R3O32AFYqbAVU
[tools] tool: [{"sku": "P-300", "name": "USB-C Hub", "price": 2490}]


[agent] assistant: В каталоге есть USB-C хаб, его цена 2490 рублей. Хотите узнать подробнее или оформить зака


## Кратковременная память через checkpointer

По умолчанию граф «забывает» всё между вызовами: каждый `invoke` начинается с чистого состояния. **Checkpointer** — это хранилище, куда LangGraph сохраняет состояние графа после каждого шага и откуда восстанавливает его при следующем вызове. Так появляется память диалога (тема 6).

- `MemorySaver()` (из `langgraph.checkpoint.memory`) — простейший checkpointer, держащий состояние **в оперативной памяти** процесса. (В продакшене берут постоянные варианты — например, на SQLite/Postgres.)
- `g.compile(checkpointer=MemorySaver())` — компилируем тот же граф, но уже с подключённым хранилищем.
- `config = {"configurable": {"thread_id": "dialog-1"}}` — обязательный конфиг при работе с checkpointer. **`thread_id`** — идентификатор *ветки диалога*: вызовы с одним и тем же `thread_id` делят одно сохранённое состояние (продолжают разговор), с разными — независимы (разные пользователи/сессии). Конфиг передаётся вторым аргументом в `invoke`.

Поскольку у `MessagesState` reducer *дополняет* список сообщений, второй запрос видит всю историю первого — поэтому агент «помнит» имя и заказ, названные ранее.

In [5]:
from langgraph.checkpoint.memory import MemorySaver

mem_graph = g.compile(checkpointer=MemorySaver())
cfg = {"configurable": {"thread_id": "dialog-1"}}

r1 = mem_graph.invoke({"messages": [("user", "Меня зовут Иван, интересует заказ ORD-1002")]}, cfg)
print("Ход 1:", r1["messages"][-1].content[:150])
r2 = mem_graph.invoke({"messages": [("user", "Как меня зовут и какой статус у того заказа?")]}, cfg)
print("Ход 2:", r2["messages"][-1].content[:200])

Ход 1: Здравствуйте, Иван! Ваш заказ ORD-1002 содержит 2 единицы товара с артикулом P-300. Статус заказа — отправлен. Если вам нужна дополнительная информаци


Ход 2: Вас зовут Иван, а статус вашего заказа ORD-1002 — отправлен. Если нужна еще какая-то информация, обращайтесь!


## Итоги

- **`@tool`** формирует инструмент (имя, описание, схема) из функции — без ручной JSON Schema.
- **`create_agent`** даёт готового ReAct-агента; **`StateGraph` + `ToolNode` + `tools_condition`** показывают его устройство (вершины, рёбра, состояние, цикл).
- **checkpointer** (`MemorySaver`) + `thread_id` — кратковременная память диалога средствами LangGraph.
- Это тот же цикл «модель ⇄ инструменты», что в базовых примерах, но фреймворк берёт рутину на себя; выбор «SDK vs фреймворк» — инженерный компромисс между прозрачностью и скоростью разработки.

### Справочник: что использовали из LangChain / LangGraph

| Имя | Библиотека | Что делает |
|-----|-----------|-----------|
| `ChatOpenAI(model, base_url, api_key, temperature)` | LangChain | объект модели (OpenAI-совместимый эндпоинт) |
| `llm.invoke(messages)` | LangChain | один вызов модели → ответ (`AIMessage`) |
| `llm.bind_tools(tools)` | LangChain | модель, «знающая» инструменты и умеющая их запрашивать |
| `@tool` | LangChain | превращает функцию в инструмент (имя/описание/схема автоматически) |
| `create_agent(model, tools, system_prompt=…)` | LangChain | готовый ReAct-агент (возвращает граф) |
| `SystemMessage` / `HumanMessage` / `ToolMessage` | LangChain | сообщения по ролям (system / user / результат инструмента) |
| `StateGraph(StateSchema)` | LangGraph | строитель графа поверх схемы состояния |
| `MessagesState` | LangGraph | готовая схема состояния `{messages: list}` с reducer-дополнением |
| `.add_node(name, fn)` | LangGraph | добавить вершину (шаг) |
| `.add_edge(A, B)` | LangGraph | безусловный переход `A → B` |
| `.add_conditional_edges(node, router)` | LangGraph | ветвление: `router` выбирает следующую вершину |
| `START`, `END` | LangGraph | точки входа и выхода графа |
| `ToolNode(tools)` | LangGraph | готовая вершина: исполняет запрошенные инструменты |
| `tools_condition` | LangGraph | готовый маршрутизатор: «есть вызов инструмента? → tools, иначе → END» |
| `.compile(checkpointer=…)` | LangGraph | собрать исполняемый граф (опц. с памятью) |
| `.invoke(input, config)` / `.stream(input)` | LangGraph | запуск целиком / с пошаговыми обновлениями |
| `MemorySaver()` | LangGraph | checkpointer: хранит состояние между вызовами (память) |
| `config={"configurable":{"thread_id":…}}` | LangGraph | идентификатор ветки диалога для checkpointer |